# ConvNeXtV2-Tiny + Pontryagin Embedding — Sugarcane Leaf Disease Classification

```
0. GPU check
1. Mount Google Drive
2. Configuration  ← EDIT paths & Kaggle credentials here
3. Clone repo + install deps + configure Python path
4. Dataset (Kaggle download → Drive cache)
5. HPO  (optional, ~1-3 h per model on T4)
6. Full training
7. Interpretability  (ScoreCAM + LIME + embedding energy)
8. Results summary
9. Zip & download
```

> **Only cell 2 needs to be edited before running.**

## 0 · GPU check

In [ ]:
import torch
print('CUDA available :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU            :', torch.cuda.get_device_name(0))
    print('VRAM (GB)      :', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1))
else:
    print('WARNING: no GPU — Runtime → Change runtime type → GPU')

## 1 · Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Configuration  ← EDIT THIS CELL

Set your Drive paths and Kaggle credentials here.  
Everything else runs automatically.

In [ ]:
import os

# ── Repository ─────────────────────────────────────────────────────────────
REPO_URL = 'https://github.com/craljimenez/rr-pontryagin-embedded.git'
REPO_DIR = '/content/pontryagin'

# ── Google Drive paths ─────────────────────────────────────────────────────
# Dataset will be cached here (survives Colab disconnects)
DRIVE_DATA_ROOT    = '/content/drive/MyDrive/____UTP/Research/PHd/RFF_activation/sugarcane_leaf_disease'
# All model checkpoints, metrics and figures go here
DRIVE_RESULTS_ROOT = '/content/drive/MyDrive/____UTP/Research/PHd/___RR_Pontryagin_embedded/results/cls_sugarcane'

# ── Kaggle credentials ─────────────────────────────────────────────────────
# Recommended: add KAGGLE_USERNAME and KAGGLE_KEY in the 🔑 Secrets panel
# (left sidebar → Secrets) so you never paste credentials in code.
try:
    from google.colab import userdata
    os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
    os.environ['KAGGLE_KEY']      = userdata.get('KAGGLE_KEY')
    print('Kaggle credentials loaded from Colab Secrets.')
except Exception:
    os.environ['KAGGLE_USERNAME'] = 'your_kaggle_username'  # ← EDIT if not using Secrets
    os.environ['KAGGLE_KEY']      = 'your_kaggle_api_key'   # ← EDIT if not using Secrets
    print('Kaggle credentials set manually.')

# ──────────────────────────────────────────────────────────────────────────
os.makedirs(DRIVE_DATA_ROOT,    exist_ok=True)
os.makedirs(DRIVE_RESULTS_ROOT, exist_ok=True)
print('Config OK.')

## 3 · Clone repo + install deps + configure Python path

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull --rebase
print(f'Repo at: {REPO_DIR}')

In [ ]:
!pip install -e {REPO_DIR} -q
!pip install scikit-optimize timm kagglehub python-dotenv lime scikit-image -q
print('Dependencies installed.')

In [ ]:
# Configure Python path — run this cell once; all subsequent cells can import freely.
import sys
for p in [f'{REPO_DIR}/experiments', f'{REPO_DIR}/src']:
    if p not in sys.path:
        sys.path.insert(0, p)

# Smoke-test the imports
from run_cls_sugarcane import _find_dataset_root, SugarcaneLeafDataset, build_dataloaders
from configs.cls_sugarcane import AVAILABLE_MODELS
print('Imports OK. Available models:', AVAILABLE_MODELS)

## 4 · Dataset

Downloads `nirmalsankalana/sugarcane-leaf-disease-dataset` via Kaggle and copies  
the class folders to Drive so the next session skips the download.

In [ ]:
import shutil
from pathlib import Path

DRIVE_DATA_PATH = Path(DRIVE_DATA_ROOT)

def _has_images(p: Path) -> bool:
    """True if p contains at least one class sub-folder with image files."""
    if not p.is_dir():
        return False
    return any(
        f.suffix.lower() in {'.jpg', '.jpeg', '.png'}
        for sub in p.iterdir() if sub.is_dir()
        for f in sub.iterdir()
    )

if _has_images(DRIVE_DATA_PATH):
    DATA_ROOT = str(DRIVE_DATA_PATH)
    print(f'Dataset already on Drive: {DATA_ROOT}')
else:
    print('Downloading from Kaggle …')
    import kagglehub
    raw_path     = Path(kagglehub.dataset_download('nirmalsankalana/sugarcane-leaf-disease-dataset'))
    dataset_root = _find_dataset_root(raw_path)
    print(f'  Downloaded to: {dataset_root}')

    print('  Copying to Drive for persistence …')
    if DRIVE_DATA_PATH.exists():
        shutil.rmtree(DRIVE_DATA_PATH)
    shutil.copytree(dataset_root, DRIVE_DATA_PATH)
    DATA_ROOT = str(DRIVE_DATA_PATH)
    print(f'  Cached at: {DATA_ROOT}')

print()

In [ ]:
from pathlib import Path

# Class distribution
print('Class distribution:')
for cls_dir in sorted(Path(DATA_ROOT).iterdir()):
    if cls_dir.is_dir():
        n = sum(1 for f in cls_dir.iterdir() if f.suffix.lower() in {'.jpg', '.jpeg', '.png'})
        print(f'  {cls_dir.name:<18} {n:>4} images')

# Build loaders and show split sizes
DATA = build_dataloaders(Path(DATA_ROOT))
print(f'\nSplits  train={len(DATA["train_ds"])}  val={len(DATA["val_ds"])}  test={len(DATA["test_ds"])}')
print(f'Classes ({DATA["n_classes"]}): {DATA["classes"]}')

## 5 · HPO *(optional — ~1-3 h per model on T4)*

Skips automatically if `best_params.json` already exists on Drive.  
You can jump straight to **section 6** to train with default hyperparameters.

In [ ]:
import json
from pathlib import Path

for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'hpo' / 'best_params.json'
    if p.exists():
        d     = json.loads(p.read_text())
        score = d.get('_best_weighted_f1', float('nan'))
        print(f'  {mt:<12} HPO done  —  best weighted_f1 = {score:.4f}')
    else:
        print(f'  {mt:<12} no HPO results yet')

In [ ]:
from pathlib import Path
if not (Path(DRIVE_RESULTS_ROOT) / 'euclidean' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_cls_sugarcane.py \
        --model        euclidean \
        --n-calls      20 \
        --n-random     8 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('euclidean HPO already done — skipping.')

In [ ]:
from pathlib import Path
if not (Path(DRIVE_RESULTS_ROOT) / 'pontryagin' / 'hpo' / 'best_params.json').exists():
    !python {REPO_DIR}/experiments/tune_cls_sugarcane.py \
        --model        pontryagin \
        --n-calls      30 \
        --n-random     10 \
        --trial-epochs 8 \
        --data-root    "{DATA_ROOT}" \
        --results-dir  "{DRIVE_RESULTS_ROOT}"
else:
    print('pontryagin HPO already done — skipping.')

## 6 · Full training

Loads `best_params.json` automatically if HPO was run. Otherwise uses config defaults.

In [ ]:
!python {REPO_DIR}/experiments/run_cls_sugarcane.py \
    --model       euclidean \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

In [ ]:
!python {REPO_DIR}/experiments/run_cls_sugarcane.py \
    --model       pontryagin \
    --data-root   "{DATA_ROOT}" \
    --results-dir "{DRIVE_RESULTS_ROOT}"

## 7 · Interpretability

- **ScoreCAM** — per-class activation heatmaps
- **LIME** — superpixel attribution maps
- **Embedding energy** — positive / negative Pontryagin subspace *(pontryagin only)*

Adjust `--n-samples` to limit the number of test images analysed.

In [ ]:
!python {REPO_DIR}/experiments/interpretability_cls_sugarcane.py \
    --model        euclidean \
    --n-samples    30 \
    --lime-samples 500 \
    --data-root    "{DATA_ROOT}" \
    --results-dir  "{DRIVE_RESULTS_ROOT}"

In [ ]:
!python {REPO_DIR}/experiments/interpretability_cls_sugarcane.py \
    --model        pontryagin \
    --n-samples    30 \
    --lime-samples 500 \
    --data-root    "{DATA_ROOT}" \
    --results-dir  "{DRIVE_RESULTS_ROOT}"

## 8 · Results summary

In [ ]:
import json
import pandas as pd
from pathlib import Path

rows = []
for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'metrics_test.json'
    if p.exists():
        m = json.loads(p.read_text())
        rows.append({
            'model':       mt,
            'accuracy':    round(m['accuracy'],    4),
            'macro_f1':    round(m['macro_f1'],    4),
            'weighted_f1': round(m['weighted_f1'], 4),
        })
    else:
        rows.append({'model': mt, 'accuracy': '—', 'macro_f1': '—', 'weighted_f1': '—'})

print('=== Test-set metrics ===')
print(pd.DataFrame(rows).set_index('model').to_string())

In [ ]:
import json
from pathlib import Path

for mt in AVAILABLE_MODELS:
    p = Path(DRIVE_RESULTS_ROOT) / mt / 'metrics_test.json'
    if not p.exists():
        print(f'{mt}: no results yet'); continue
    m = json.loads(p.read_text())
    print(f'\n{mt.upper()} — per-class:')
    for cls, v in m['per_class'].items():
        print(f"  {cls:<18}  F1={v['f1']:.4f}  prec={v['precision']:.4f}  "
              f"rec={v['recall']:.4f}  n={int(v['support'])}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
for ax, mt in zip(axes, AVAILABLE_MODELS):
    img_path = Path(DRIVE_RESULTS_ROOT) / mt / 'viz' / 'confusion_matrix.png'
    if img_path.exists():
        ax.imshow(mpimg.imread(str(img_path)))
        ax.set_title(mt, fontsize=13, fontweight='bold')
    else:
        ax.text(0.5, 0.5, 'not available', ha='center', va='center', fontsize=12)
    ax.axis('off')
plt.suptitle('Confusion matrices — test set', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Display ScoreCAM, LIME and embedding energy PDFs inline.
# Converts PDFs to PNGs with poppler (installed below if needed).
import subprocess
subprocess.run(['apt-get', 'install', '-y', '-q', 'poppler-utils'], check=True)

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

def _show_pdfs(folder: Path, title_prefix: str, n: int = 3):
    if not folder.exists():
        print(f'  {folder} not found — skip'); return
    for pdf in sorted(folder.glob('*.pdf'))[:n]:
        png = pdf.with_suffix('.png')
        subprocess.run(
            ['pdftoppm', '-r', '120', '-png', '-singlefile', str(pdf), str(png.with_suffix(''))],
            check=False, capture_output=True,
        )
        if png.exists():
            fig, ax = plt.subplots(figsize=(16, 4))
            ax.imshow(mpimg.imread(str(png)))
            ax.set_title(f'{title_prefix} — {pdf.stem}', fontsize=10)
            ax.axis('off')
            plt.tight_layout()
            plt.show()

root = Path(DRIVE_RESULTS_ROOT)
for mt in AVAILABLE_MODELS:
    print(f'\n── {mt.upper()} ──────────────────────')
    _show_pdfs(root / mt / 'scorecam',         f'ScoreCAM [{mt}]')
    _show_pdfs(root / mt / 'lime',             f'LIME [{mt}]',    n=2)

print('\n── PONTRYAGIN — Embedding energy ──────────')
_show_pdfs(root / 'pontryagin' / 'embedding_energy', 'Embedding energy')

## 9 · Zip & download results

The Drive copy already persists. This cell is for downloading to your local machine.

In [ ]:
import shutil
from google.colab import files
from pathlib import Path

zip_path = '/content/cls_sugarcane_results'
shutil.make_archive(zip_path, 'zip', DRIVE_RESULTS_ROOT)
size_mb = Path(zip_path + '.zip').stat().st_size / 1e6
print(f'Archive: {zip_path}.zip  ({size_mb:.1f} MB)')
files.download(zip_path + '.zip')

In [ ]:
# Browse the Drive results tree
import os
for root, dirs, fnames in os.walk(DRIVE_RESULTS_ROOT):
    level  = root.replace(DRIVE_RESULTS_ROOT, '').count(os.sep)
    indent = '  ' * level
    print(f'{indent}{os.path.basename(root)}/')
    if level < 3:
        for f in sorted(fnames):
            print(f'{indent}  {f}')